# Configuración del entorno

Procesamiento digital de señales con el sensor WT9011DCL

Antes de trabajar con el sensor revisamos que la computadora tenga lo necesario: la versión de Python, las librerías del curso, que las gráficas se muestren en el notebook y que el Bluetooth detecte el sensor. Al final se crean las carpetas donde se guardarán las grabaciones.

El kernel del notebook debe ser Python (PDS WT9011). Si no aparece en la lista, seleccionar el intérprete `.venv\Scripts\python.exe` de esta carpeta.

## Versión de Python

Se requiere Python 3.10 o superior y que el notebook use el entorno virtual `.venv` del curso.

In [ ]:
import sys
import platform

print("Python:    ", sys.version.split()[0])
print("Ejecutable:", sys.executable)
print("Sistema:   ", platform.system(), platform.release())

assert sys.version_info >= (3, 10), "Se necesita Python 3.10 o superior"

if ".venv" not in sys.executable:
    print("\nEl notebook no está usando el entorno .venv del curso, cambiar el kernel.")

## Librerías

| Librería | Uso en el curso |
|---|---|
| bleak | comunicación Bluetooth Low Energy con el sensor |
| numpy | arreglos y cálculo numérico |
| scipy | filtros, FFT y densidad espectral |
| pandas | tablas de datos y archivos CSV |
| matplotlib | gráficas |
| scikit-learn | escalado y división en entrenamiento y prueba |

Si alguna aparece como faltante, quitar el comentario de la línea `%pip install` en la celda siguiente, ejecutarla y reiniciar el kernel.

In [ ]:
# %pip install -r requirements.txt

from importlib.metadata import version, PackageNotFoundError

paquetes = ["bleak", "numpy", "scipy", "pandas", "matplotlib", "scikit-learn"]

for p in paquetes:
    try:
        print(f"{p:<14} {version(p)}")
    except PackageNotFoundError:
        print(f"{p:<14} falta instalar")

## Gráficas

Para comprobar que matplotlib funciona se grafica una senoidal de 5 Hz muestreada a 100 Hz. Los puntos son las muestras, que es lo único que registra la computadora; la línea solo las une.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fs = 100                      # frecuencia de muestreo [Hz]
f = 5                         # frecuencia de la señal [Hz]
t = np.arange(0, 1, 1 / fs)   # 1 s de tiempo
x = np.sin(2 * np.pi * f * t)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, x, lw=1)
ax.plot(t, x, ".", ms=5)
ax.set(xlabel="Tiempo [s]", ylabel="Amplitud", title=f"Senoidal de {f} Hz, fs = {fs} Hz")
ax.grid(alpha=0.3)
plt.show()

## Detección del sensor por Bluetooth

El WT9011DCL transmite por Bluetooth Low Energy (BLE). Con el sensor encendido, la celda escanea durante 8 segundos y muestra los sensores WitMotion encontrados, que se anuncian con un nombre que empieza con WT.

La dirección es el identificador único de cada sensor y es la que se usa para conectarse. El RSSI es la intensidad de la señal en dBm: un valor cercano a -40 indica que el sensor está muy cerca y uno cercano a -90, que está lejos. Como los dos sensores del salón pueden tener el mismo nombre, conviene acercar el propio a la computadora y quedarse con la dirección de mayor RSSI.

Si no aparece, revisar que el Bluetooth esté encendido, que el sensor tenga batería y que no esté conectado a otro equipo o a la aplicación del celular; un sensor BLE acepta una sola conexión a la vez.

In [ ]:
from bleak import BleakScanner

MOSTRAR_TODOS = False   # True para listar también los demás dispositivos BLE

encontrados = await BleakScanner.discover(timeout=8.0, return_adv=True)
dispositivos = sorted(encontrados.values(), key=lambda par: par[1].rssi, reverse=True)

print(f"{'Nombre':<30}{'Dirección':<20}{'RSSI':>6}")
n_wt = 0
for d, adv in dispositivos:
    nombre = d.name or adv.local_name or "(sin nombre)"
    es_wt = nombre.upper().startswith("WT")
    n_wt += es_wt
    if es_wt or MOSTRAR_TODOS:
        print(f"{nombre[:28]:<30}{d.address:<20}{adv.rssi:>6}")

print(f"\nDispositivos BLE detectados: {len(dispositivos)}. Sensores WitMotion: {n_wt}.")

## Carpetas de datos

`datos/demo` guarda grabaciones de ejemplo para seguir la clase aunque falle la conexión con el sensor. `datos/equipo_A` y `datos/equipo_B` guardan las grabaciones de cada equipo.

In [ ]:
from pathlib import Path

for carpeta in ["datos/demo", "datos/equipo_A", "datos/equipo_B"]:
    Path(carpeta).mkdir(parents=True, exist_ok=True)
    print(Path(carpeta).resolve())

Antes de pasar a `01_conexion` hay que anotar la dirección del sensor del equipo.